# Solution C2 · Three outliers out of 303 points hijack the first component

**Type**: Break the model

## Objective

Add three anomalous points to a cloud of 300 points and measure how much the first principal component rotates (by 90 degrees: it moves from one column to another). Fix with a clipping (winsorizing) of the extreme values and verify that
the direction goes back to the true one.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.decomposition import PCA

rng = np.random.default_rng(1)
A = rng.multivariate_normal([0, 0, 0, 0], np.diag([9, 4, 1, 0.5]), 300)
outliers = np.array([[0, 0, 60, 0], [0, 0, -55, 5], [0, 0, 58, -3]])
B = np.r_[A, outliers]
print(A.shape, B.shape)

## Commented solution

In [ ]:
pca_A, pca_B = PCA().fit(A), PCA().fit(B)
pc1_A, pc1_B = pca_A.components_[0], pca_B.components_[0]
A_column, B_column = int(np.abs(pc1_A).argmax()), int(np.abs(pc1_B).argmax())
angle_degrees = float(np.degrees(np.arccos(min(1, abs(pc1_A @ pc1_B)))))
ratio_A, ratio_B = pca_A.explained_variance_ratio_[0], pca_B.explained_variance_ratio_[0]
outlier_share = 100 * 3 / len(B)
print(f"column with the largest loading in PC1: {A_column} -> {B_column} | angle between the two PC1: {angle_degrees:.0f} degrees")
print(f"variance explained by PC1: {ratio_A:.3f} -> {ratio_B:.3f} | outliers: {outlier_share:.1f}% of the points")

In [ ]:
lo, hi = np.percentile(B, [1, 99], axis=0)
B_clip = np.clip(B, lo, hi)
pc1_clip = PCA().fit(B_clip).components_[0]
clip_column = int(np.abs(pc1_clip).argmax())
clip_angle_degrees = float(np.degrees(np.arccos(min(1, abs(pc1_clip @ pc1_A)))))
print(f"after the clipping: PC1 on column {clip_column}, angle with the true PC1 {clip_angle_degrees:.1f} degrees")

In [ ]:
d = B - A.mean(0)
mahalanobis2 = np.einsum("ij,jk,ik->i", d, np.linalg.inv(np.cov(A.T)), d)
outlier_indices = list(np.argsort(mahalanobis2)[-3:])
print("points with the largest Mahalanobis distance:", sorted(outlier_indices), "| distances^2:", np.sort(mahalanobis2)[-3:].round(0))

## Check

In [ ]:
with verify():
    check(A_column == 0 and B_column == 2, "without outliers PC1 is column 0; with the three outliers it is column 2", "np.abs(pc1).argmax()")
    check(angle_degrees > 85 and close(outlier_share, 100 * 3 / 303), "PC1 rotates by about 90 degrees because of three points (1% of the data)", "arccos(|pc1_A . pc1_B|)")
    check(ratio_B > ratio_A and ratio_A < 0.65 and ratio_B > 0.7, "the variance explained by PC1 goes from 0.59 to 0.73: misleading", "explained_variance_ratio_[0]")
    check(B_clip.shape == B.shape and clip_column == 0 and clip_angle_degrees < 15, "after the clipping PC1 is column 0 again (angle with the true one below 15 degrees)", "np.clip(column, percentile 1, percentile 99) for every column")
    check(sorted(outlier_indices) == [300, 301, 302], "the Mahalanobis distance identifies the three outliers (indices 300, 301, 302)", "use the covariance of A and the mean of A")
done()

## What you learned

- **Three points out of 303 (1%) rotate PC1 by almost 90 degrees**: PCA maximizes the variance, and a point at distance 60 contributes $60^2 = 3600$ to the variance of a column whose true variance is 1. The "principal" component simply becomes the direction of the outliers.
- **The symptom is misleading:** the share of variance explained by PC1 goes from 0.59 to 0.73: it looks like the data are simpler than before, when they are just dirtier. A PCA that "works better" after the addition of suspicious data is a warning.
- Remedies: **clip** the extreme values (winsorizing, as here), remove the outliers identified with the Mahalanobis distance or an `EllipticEnvelope`, use a **robust PCA** (based on robust covariance estimators like `MinCovDet`). The order matters: the outliers must be handled **before** PCA and learned on the training set alone.
- The Mahalanobis distance identifies isolated outliers well; with many outliers the covariance itself estimated with the dirty data gets contaminated (masking): that is why here it is computed with `A`, that is with data that we assume clean.

## Common pitfall

Applying PCA to uninspected data and interpreting the first component as "the main factor". Look at the outliers first (box plot, Mahalanobis distances) and compare PC1 with and without the extreme points.

## Interview question

> *Click on a question to show its answer.*

<details><summary>Is PCA sensitive to outliers?</summary>

Very: it is based on variance and covariance, that is on squares, so a few distant points can hijack the first component. I inspect the outliers first (Mahalanobis, isolation forest), clip or remove them on the training set alone, or use a robust version (MCD covariance, robust PCA).
I always check that the first components are stable with a bootstrap or by removing a few points.

</details>